# rotation-matrix-3d-y-axis — ex7: compose Rx · Ry · Rz on a cube + 3-D scatter

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rotation-matrix-3d-y-axis`. Running the final beacon cell reports progress against the `Numpy: Applied patterns and advanced` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rotation-matrix-3d-y-axis`** (exercise 7). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rotation-matrix-3d-y-axis"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Y-axis rotation — quick refresher

**The matrix.** Right-hand rotation by `θ` about Y:
```
R_y(θ) = [[ cos θ,  0,  sin θ],
          [ 0,      1,  0    ],
          [-sin θ,  0,  cos θ]]
```
Anything along Y stays put (middle row `[0,1,0]`); the X-Z plane rotates.

**Acting on data.** Column-vector form: `v' = R @ v`. Batch of row-vectors `(N, 3)`: `points' = points @ R.T`. Composition: `R(α) @ R(β) = R(α + β)` for single-axis rotations; multi-axis rotations don't commute.

**Numerical truth.** Rotation matrices are orthogonal: `R @ R.T = I` and `R.inverse() == R.T`. Floating-point composition accumulates ~1e-7 error per matmul.

### Exercise 7 — compose Rx · Ry · Rz on a cube + 3-D scatter

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Compose three single-axis rotations into a multi-axis transform and apply it to a cube's 8 vertices, then plot the original and rotated cubes in 3-D.
> Keywords: multi-axis-rotation, composition, cube, 3d-scatter, integrative
> ```

**KCs targeted:** `rotation-matrix-y-construct`, `rotation-composition-multi-axis`, `rotate-batch-of-points`

Implement `ex7_compose_xyz(points, ax, ay, az)`. Given a `(N, 3)` batch of points and three scalar angles, build the rotation matrices `R_x(ax)`, `R_y(ay)`, `R_z(az)` and return the rotated points `points @ (R_x @ R_y @ R_z).T`.

**The three matrices** (right-hand rule):
```
R_x(θ) = [[1, 0, 0], [0, c, -s], [0, s, c]]
R_y(θ) = [[c, 0, s], [0, 1, 0], [-s, 0, c]]
R_z(θ) = [[c, -s, 0], [s, c, 0], [0, 0, 1]]
```
where `c = cos θ`, `s = sin θ` for each axis's angle.

Apply order matters: `R_x @ R_y @ R_z` is **not** equal to `R_z @ R_y @ R_x`. Use the order given.

The test (1) checks all-zero angles act as identity, (2) checks distances from origin are preserved, (3) checks a hand-computed single-axis case, then plots the original cube vertices vs the rotated cube in a 3-D scatter.

In [ ]:
def ex7_compose_xyz(points: Tensor, ax: float, ay: float, az: float) -> Tensor:
    """Apply R_x(ax) @ R_y(ay) @ R_z(az) to (N, 3) points. Returns (N, 3)."""
    raise NotImplementedError()


def _test_ex7():
    import math

    # 8 cube vertices at the corners of [-1, 1]^3.
    cube = t.tensor([
        [-1.0, -1.0, -1.0], [ 1.0, -1.0, -1.0],
        [-1.0,  1.0, -1.0], [ 1.0,  1.0, -1.0],
        [-1.0, -1.0,  1.0], [ 1.0, -1.0,  1.0],
        [-1.0,  1.0,  1.0], [ 1.0,  1.0,  1.0],
    ])

    # All zeros → identity.
    out0 = ex7_compose_xyz(cube, 0.0, 0.0, 0.0)
    assert out0.shape == (8, 3), f'expected (8,3), got {tuple(out0.shape)}'
    assert t.allclose(out0, cube, atol=1e-6), 'zero angles should not move points'

    # Distance preservation: all vertices stay at sqrt(3) from origin.
    out = ex7_compose_xyz(cube, 0.4, -0.7, 1.1)
    norms = out.pow(2).sum(dim=-1).sqrt()
    assert t.allclose(norms, t.full((8,), math.sqrt(3.0)), atol=1e-5), \
        f'cube vertices should stay at sqrt(3) from origin, got {norms}'

    # Single-axis check: only Y rotation by π/2.
    out_y = ex7_compose_xyz(cube, 0.0, math.pi / 2, 0.0)
    # R_y(π/2): X → -Z, Z → +X, Y unchanged.
    # So vertex (1, 1, 1) → (1, 1, -1)? Let's compute: c=0, s=1.
    # R_y @ [1,1,1] = [c*1 + s*1, 1, -s*1 + c*1] = [1, 1, -1].
    vert_111 = out_y[7]
    assert t.allclose(vert_111, t.tensor([1.0, 1.0, -1.0]), atol=1e-6), \
        f'(1,1,1) under R_y(π/2) should be (1,1,-1), got {vert_111}'

    # Non-commutativity smoke check.
    out_xyz = ex7_compose_xyz(cube, 0.5, 0.5, 0.5)
    # Compare against a manual zyx ordering to confirm we built xyz, not zyx.
    ax, ay, az = 0.5, 0.5, 0.5
    cx, sx = math.cos(ax), math.sin(ax)
    cy, sy = math.cos(ay), math.sin(ay)
    cz, sz = math.cos(az), math.sin(az)
    Rx = t.tensor([[1, 0, 0], [0, cx, -sx], [0, sx, cx]], dtype=t.float32)
    Ry = t.tensor([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]], dtype=t.float32)
    Rz = t.tensor([[cz, -sz, 0], [sz, cz, 0], [0, 0, 1]], dtype=t.float32)
    ref_xyz = cube @ (Rx @ Ry @ Rz).T
    assert t.allclose(out_xyz, ref_xyz, atol=1e-5), \
        f'composition order mismatch:\n{out_xyz}\nvs\n{ref_xyz}'

    # 3-D scatter: original cube (gray) vs rotated cube (color).
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers projection)
    fig = plt.figure(figsize=(6, 5.5))
    ax_ = fig.add_subplot(111, projection='3d')
    ax_.scatter(cube[:, 0], cube[:, 1], cube[:, 2], c='lightgray', s=60, label='original')
    ax_.scatter(out[:, 0], out[:, 1], out[:, 2], c=range(8), cmap='plasma', s=80, label='rotated')
    for i in range(8):
        ax_.plot([cube[i, 0], out[i, 0]],
                 [cube[i, 1], out[i, 1]],
                 [cube[i, 2], out[i, 2]], 'k--', alpha=0.3, linewidth=0.7)
    ax_.set_title('Cube before (gray) and after R_x · R_y · R_z rotation')
    ax_.set_xlabel('X'); ax_.set_ylabel('Y'); ax_.set_zlabel('Z')
    ax_.legend(loc='upper left')
    fig.tight_layout()
    plt.show()

    print(f'rotated cube shape: {tuple(out.shape)}')
    print(f'norm preservation: max deviation = {(norms - math.sqrt(3.0)).abs().max().item():.2e}')
    _dd_passed.add('ex7')
    print("ex7 ✓")

_test_ex7()

<details><summary>Solution</summary>

```python
def ex7_compose_xyz(points: Tensor, ax: float, ay: float, az: float) -> Tensor:
    import math
    cx, sx = math.cos(ax), math.sin(ax)
    cy, sy = math.cos(ay), math.sin(ay)
    cz, sz = math.cos(az), math.sin(az)
    Rx = t.tensor([
        [1.0, 0.0, 0.0],
        [0.0, cx,  -sx],
        [0.0, sx,  cx ],
    ])
    Ry = t.tensor([
        [cy,  0.0, sy ],
        [0.0, 1.0, 0.0],
        [-sy, 0.0, cy ],
    ])
    Rz = t.tensor([
        [cz,  -sz, 0.0],
        [sz,   cz, 0.0],
        [0.0, 0.0, 1.0],
    ])
    R = Rx @ Ry @ Rz
    return points @ R.T
```

**Order matters.** `R_x @ R_y @ R_z` reads right-to-left when acting on a column vector: first rotate about Z, then about Y, then about X. Swapping the order produces a different orientation for any non-trivial angle combination. Pick one convention and stick to it across your whole codebase.

**Why we can't just "add angles".** Single-axis rotations commute (`R_y(α) @ R_y(β) = R_y(α+β)`), but multi-axis don't. This is why orientation in 3-D needs three numbers (Euler angles, axis-angle, quaternion) with an agreed-upon convention — there's no scalar shortcut.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex7',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()